## Objective: Image Preprocessing and Feature Extraction

This demonstration aims to illustrate the process of image preprocessing, specifically focusing on applying a filter, and subsequent feature extraction using Gray-Level Co-occurrence Matrix (GLCM). The objective is to understand how preprocessing can influence the texture features extracted from an image.

## Algorithm Overview

### 1. Preprocessing (Filtering)

**Goal**: To enhance image quality, remove noise, or highlight certain features before feature extraction. Filtering can smooth out textures or emphasize edges.

**Algorithm (Gaussian Blur)**:
- A Gaussian filter is a low-pass filter that blurs an image and reduces noise.
- It uses a Gaussian function to calculate the transformation to apply to each pixel in the image.
- The `sigma` parameter controls the standard deviation of the Gaussian, influencing the degree of blurring.

### 2. Feature Extraction (Gray-Level Co-occurrence Matrix - GLCM)

**Goal**: To quantify textural properties of an image by considering the spatial relationship of pixels. GLCM features provide insights into the regularity, coarseness, and contrast of textures.

**Algorithm Steps**:
1.  **Create GLCM**: For a given image, a GLCM is constructed by counting how often pairs of pixels with specific gray-level values and a specified spatial relationship (distance and angle) occur in the image.
2.  **Calculate Statistical Features**: From the GLCM, several statistical features are derived to characterize the texture. Common features include:
    *   **Contrast**: Measures the intensity contrast between a pixel and its neighbor over the whole image. A high contrast indicates a large difference in intensity.
    *   **Dissimilarity**: Similar to contrast, but it's a linear measure of how different the pixel pairs are.
    *   **Homogeneity**: Measures the closeness of the distribution of elements in the GLCM to the GLCM diagonal. High homogeneity implies smaller differences in intensity.
    *   **Energy (Angular Second Moment)**: Measures the sum of squared elements in the GLCM. High energy indicates textural uniformity.
    *   **Correlation**: Measures the linear dependency of gray levels in the image. High correlation means a strong linear relationship between pixel intensities and their neighbors.
    *   **ASM (Angular Second Moment)**: A measure of the uniformity of an image.

We will extract these features from both the original and the filtered image to compare the impact of preprocessing.

In [ ]:
# Install necessary libraries
!pip install scikit-image matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import gaussian
from skimage.color import rgb2gray

# Load a sample image
image = data.camera()

# Convert to grayscale if it's not already (GLCM works on grayscale)
if image.ndim == 3:
    image = rgb2gray(image)

# Display the original image
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

# Apply a Gaussian filter (preprocessing step)
filtered_image = gaussian(image, sigma=1.0) # sigma controls the degree of blurring

# Display the filtered image
plt.subplot(1, 2, 2)
plt.imshow(filtered_image, cmap='gray')
plt.title('Filtered Image (Gaussian Blur)')
plt.axis('off')
plt.show()

print('Original image shape:', image.shape)
print('Filtered image shape:', filtered_image.shape)

In [ ]:
# --- Extract GLCM Features from Original Image ---

# Quantize image to reduce number of gray levels (important for GLCM)
# GLCM performance can be impacted by a large number of gray levels
image_quantized = (image * 255).astype(np.uint8)

# Calculate GLCM. Parameters:
# distances: list of pixel pair distances
# angles: list of pixel pair angles (0, 45, 90, 135 degrees)
# levels: number of gray levels in the image
# symmetric=True: consider (i,j) and (j,i) pairs
# normed=True: normalize the GLCM
glcd_original = graycomatrix(image_quantized, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)

# Extract GLCM properties
properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
features_original = {p: graycoprops(glcd_original, p).mean() for p in properties}

print('GLCM Features for Original Image:')
for prop, value in features_original.items():
    print(f'- {prop}: {value:.4f}')

In [ ]:
# --- Extract GLCM Features from Filtered Image ---

# Quantize filtered image
filtered_image_quantized = (filtered_image * 255).astype(np.uint8)

# Calculate GLCM for the filtered image
glcd_filtered = graycomatrix(filtered_image_quantized, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)

# Extract GLCM properties
features_filtered = {p: graycoprops(glcd_filtered, p).mean() for p in properties}

print('GLCM Features for Filtered Image (Gaussian Blur):')
for prop, value in features_filtered.items():
    print(f'- {prop}: {value:.4f}')

In [ ]:
# --- Compare Results ---
import pandas as pd

comparison_df = pd.DataFrame({
    'Feature': properties,
    'Original Image': [features_original[p] for p in properties],
    'Filtered Image': [features_filtered[p] for p in properties]
})

print('\nComparison of GLCM Features:')
display(comparison_df.set_index('Feature'))

# You can also visualize the differences
fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(x='Feature', y=['Original Image', 'Filtered Image'], kind='bar', ax=ax)
plt.title('Comparison of GLCM Features: Original vs. Filtered Image')
plt.ylabel('Feature Value')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Conclusion

From the comparison of GLCM features between the original and Gaussian-filtered images, we can draw the following conclusions:

*   **Contrast and Dissimilarity**: These values typically *decrease* after applying a Gaussian blur. This is expected, as blurring reduces sharp intensity differences between pixels, making the image smoother and less contrasting.
*   **Homogeneity and Energy (ASM)**: These values generally *increase* with filtering. A smoother image implies more uniform regions, leading to higher homogeneity (more similar pixel pairs) and higher energy (more consistent texture patterns).
*   **Correlation**: The effect on correlation can vary, but often it might *increase* or remain similar. Blurring can make pixel relationships more linear over short distances by reducing high-frequency noise.

**Overall Impact**: Preprocessing, such as Gaussian filtering, significantly alters the textural characteristics of an image. By smoothing out fine details and noise, it changes how co-occurring pixel intensities are distributed. This highlights the importance of choosing appropriate preprocessing steps based on the specific application and the type of features one aims to extract. For instance, if the goal is to capture fine textures, heavy blurring might be counterproductive. If the goal is to generalize textures and reduce noise, filtering is beneficial.